# EcoFood — Google Trends

Notebook con tu código original, separado en celdas. Se mantienen los términos, Argentina, el período de 5 años, las pausas y los cálculos.

In [1]:
import time
import os
import pandas as pd
from pytrends.request import TrendReq

pytrends = TrendReq(hl="es-AR", tz=180)

terminos = ["alimentación saludable", "veganismo", "sin TACC",
            "café de especialidad", "alimentos naturales"]

# Ruta del CSV grupal existente
carpeta = os.path.dirname(os.getcwd())
ruta = os.path.join(carpeta, "data", "raw", "trends_juntos.csv")

## 1. Consulta conjunta

In [2]:
try:
    pytrends.build_payload(terminos, geo="AR", timeframe="today 5-y")
    juntos = pytrends.interest_over_time()
    juntos = juntos[juntos["isPartial"] == False].drop(columns="isPartial")

    juntos.index = pd.to_datetime(juntos.index)
    juntos.index.name = "fecha"
    print("Búsqueda grupal completada.")

except Exception as error:
    print("La búsqueda grupal fue interrumpida:", error)

    if os.path.exists(ruta):
        print("Se cargarán los datos del CSV grupal existente.")
        juntos = pd.read_csv(ruta, index_col="fecha", parse_dates=["fecha"])
    else:
        print("No existe un CSV grupal para mostrar.")
        juntos = pd.DataFrame()

if not juntos.empty:
    display(juntos.head())

time.sleep(30)

,alimentación saludable,veganismo,sin TACC,café de especialidad,alimentos naturales
date,,,,,
2021-11-07,9,5,50,0,3
2021-11-14,9,3,50,0,2
2021-11-21,8,3,50,0,3
2021-11-28,5,2,46,0,2
2021-12-05,4,2,49,0,1


## 2. Segunda consulta: productos orgánicos

In [ ]:
try:
    pytrends.build_payload(["productos orgánicos"], geo="AR", timeframe="today 5-y")
    organicos = pytrends.interest_over_time()

    if not organicos.empty:
        organicos = organicos[organicos["isPartial"] == False].drop(columns="isPartial")
        organicos.index = pd.to_datetime(organicos.index)
        organicos.index.name = "fecha"

        # Sumamos la columna al mismo DataFrame y al mismo CSV grupal.
        if juntos.empty:
            juntos = organicos.copy()
        else:
            if "productos orgánicos" in juntos.columns:
                juntos = juntos.drop(columns="productos orgánicos")
            juntos = juntos.join(organicos[["productos orgánicos"]], how="outer")

        juntos = juntos.sort_index()
        juntos.index.name = "fecha"
        juntos.to_csv(ruta, index=True, date_format="%Y-%m-%d")

        print("Búsqueda de productos orgánicos completada y agregada a trends_juntos.csv.")
        display(organicos)

    else:
        print("Google Trends no devolvió datos para productos orgánicos.")

except Exception as error:
    print("No se pudo completar la búsqueda de productos orgánicos:", error)
    organicos = pd.DataFrame()

time.sleep(30)

## 3. Consultas individuales

In [3]:
series = {}
for t in terminos:
    pytrends.build_payload([t], geo="AR", timeframe="today 5-y")
    d = pytrends.interest_over_time()
    if t in d:
        series[t] = d[t][d["isPartial"] == False]
    print("listo:", t)
    time.sleep(30)

indiv = pd.DataFrame(series)

display(indiv.head())

listo: alimentación saludable
listo: veganismo
listo: sin TACC
listo: café de especialidad
listo: alimentos naturales


,alimentación saludable,veganismo,sin TACC,café de especialidad,alimentos naturales
date,,,,,
2021-11-07,89,60,50,0,81
2021-11-14,90,31,50,0,61
2021-11-21,83,34,50,0,73
2021-11-28,53,27,46,0,55
2021-12-05,35,23,49,0,32


## 4. Guardado de CSV

En Jupyter no existe `__file__`, así que se usa la carpeta actual para guardar los archivos.

In [ ]:
print("CSV grupal:", ruta)

## 5. Información y resumen

In [ ]:
print("\\nInterés de búsqueda relativo (escala común, 2022-2025)")
comun = juntos.loc["2022":"2025"].drop(columns=["productos orgánicos"], errors="ignore")
print(comun.mean().sort_values(ascending=False).round(1))

print("\\nPorcentaje de semanas en zona de ruido (índice <= 2)")
print(((comun <= 2).mean() * 100).round(0))

print("\\nProductos orgánicos (consulta independiente; escala propia, 2022-2025)")
if "productos orgánicos" in juntos.columns:
    print(juntos.loc["2022":"2025", "productos orgánicos"].mean().round(1))

print("\\nDirección de cada término (escala propia, promedio por año)")
print(indiv.loc["2022":"2025"].groupby(indiv.loc["2022":"2025"].index.year).mean().round(0))